Importing libraries

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

loading preprocessed data

In [ ]:
INTERIM = Path("../data/interim")

eaf_temp        = pd.read_parquet(INTERIM / "eaf_temp.parquet")
eaf_chem        = pd.read_parquet(INTERIM / "eaf_final_chemical.parquet")
eaf_transformer = pd.read_parquet(INTERIM / "eaf_transformer.parquet")
eaf_gaslance    = pd.read_parquet(INTERIM / "eaf_gaslance.parquet")
inj_mat         = pd.read_parquet(INTERIM / "inj_mat.parquet")
basket_charged  = pd.read_parquet(INTERIM / "basket_charged.parquet")
eaf_added       = pd.read_parquet(INTERIM / "eaf_added.parquet")

### Dissolved oxygen — data cleaning

Raw `VALO2_PPM` values include two sentinel codes and a physical upper bound that must be removed before analysis:

| Value | Meaning |
|-------|---------|
| `0` | No O₂ sensor — probe equipped with thermometer only |
| `9999` | Equipment sentinel — saturation or out-of-range reading |
| `> 3400 ppm` | Physically implausible — exceeds Fe-O equilibrium limit at 1700 °C |

All three cases are replaced with `NA` prior to any further use of this variable.

In [ ]:
O2_SENTINEL_VALUES = {0, 9999}
O2_MAX_PHYSICAL = 3400

eaf_temp["VALO2_PPM"] = eaf_temp["VALO2_PPM"].where(
    ~eaf_temp["VALO2_PPM"].isin(O2_SENTINEL_VALUES)
    & eaf_temp["VALO2_PPM"].lt(O2_MAX_PHYSICAL),
    other=pd.NA,
)

### Temperature and oxygen distributions

Histograms of steel bath temperature and dissolved oxygen content measured at tapping.
Vertical reference lines mark the physically reasonable operating range for EAF steelmaking.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(eaf_temp["TEMP"].dropna(), bins=60, edgecolor="white", linewidth=0.4)
axes[0].set(title="Steel temperature", xlabel="Temperature (°C)", ylabel="Count")
axes[0].axvline(1500, color="red", linestyle="--", label="Min. threshold (~1500°C)")
axes[0].axvline(1720, color="orange", linestyle="--", label="Max. threshold (~1720°C)")
axes[0].legend(fontsize=8)

axes[1].hist(eaf_temp["VALO2_PPM"].dropna(), bins=60, edgecolor="white", linewidth=0.4)
axes[1].set(title="Dissolved oxygen in steel", xlabel="Dissolved Oxygen (ppm)", ylabel="Count")

plt.tight_layout()
plt.show()

### Oxygen and gas injection distribution

The linear histograms below suggest clean distributions, but a log-scale y-axis can reveal
low-frequency outliers hidden in the tail. Reference lines mark the thresholds above which
values are considered suspect for normal EAF operation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(eaf_gaslance["O2_AMOUNT"].dropna(), bins=60, edgecolor="white", linewidth=0.4)
axes[0].set(title="Oxygen injection — linear scale", xlabel="Volume (Nm³)", ylabel="Count")

axes[1].hist(eaf_gaslance["O2_AMOUNT"].dropna(), bins=60, edgecolor="white", linewidth=0.4)
axes[1].set_yscale("log")
axes[1].set(title="Oxygen injection — log scale", xlabel="Volume (Nm³)", ylabel="Count (log)")
axes[1].axvline(10_000, color="red", linestyle="--", linewidth=1, label="Suspect threshold (10k Nm³)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(eaf_gaslance["GAS_AMOUNT"].dropna(), bins=60, edgecolor="white", linewidth=0.4)
axes[0].set(title="Gas injection — linear scale", xlabel="Volume (Nm³)", ylabel="Count")

axes[1].hist(eaf_gaslance["GAS_AMOUNT"].dropna(), bins=60, edgecolor="white", linewidth=0.4)
axes[1].set_yscale("log")
axes[1].set(title="Gas injection — log scale", xlabel="Volume (Nm³)", ylabel="Count (log)")
axes[1].axvline(2_500, color="red", linestyle="--", linewidth=1, label="Suspect threshold (2.5k Nm³)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print("=== O2_AMOUNT > 10.000 Nm³ ===")
high_o2 = eaf_gaslance[eaf_gaslance["O2_AMOUNT"] > 10_000]
print(f"Total records: {len(high_o2):,}")
print(f"Unique HEATIDs:     {high_o2['HEATID'].nunique():,}")
print(f"Records per HEATID (top 10):")
print(high_o2.groupby("HEATID").size().sort_values(ascending=False).head(10))

print("\n=== GAS_AMOUNT > 2.500 Nm³ ===")
high_gas = eaf_gaslance[eaf_gaslance["GAS_AMOUNT"] > 2_500]
print(f"Total records: {len(high_gas):,}")
print(f"Unique HEATIDs:     {high_gas['HEATID'].nunique():,}")
print(f"Records per HEATID (top 10):")
print(high_gas.groupby("HEATID").size().sort_values(ascending=False).head(10))

### Diagnosis: anomalous injection records concentrated in four heats

Records with physically implausible O₂ or gas volumes are not randomly distributed —
they are concentrated in four HEATIDs: `7F5591`, `6F4696`, `7F5592`, and `6F4697`.
Heat `7F5591` alone accounts for over 88% of anomalous O₂ records and over 99% of anomalous gas records,
which is consistent with a SCADA accumulation bug affecting a specific equipment session rather than
a systematic sensor fault.

**Decision:** these four heats will be excluded from gas lance features in the feature engineering notebook.
They are retained in the main dataset as they may contain valid readings in other data sources.

### Temperature measurement coverage per heat

The immersion probe is inserted into the steel bath at specific points during the heat —
typically after each scrap basket has melted and mandatorily before tapping. The modal
pattern of 3–5 measurements per heat is consistent with a two or three-basket EAF operation:
one reading after the first basket, one after the second, and a final reading immediately
before tapping.

The 1055 heats with only one measurement are an ambiguous case. That single reading may
correspond to the tapping measurement of a short, uneventful heat — or it may be an
intermediate reading from a heat that was aborted before tapping. The two cases are
indistinguishable from measurement count alone, which introduces uncertainty in target quality.

**Decision:** these heats are retained for now. If they systematically concentrate higher
prediction errors, they will be revisited and potentially excluded during the modeling step.

In [ ]:
meas_per_heat = eaf_temp.groupby("HEATID").size()

fig, ax = plt.subplots(figsize=(10, 4))
meas_per_heat.clip(upper=20).value_counts().sort_index().plot(kind="bar", ax=ax, edgecolor="white")
ax.set(
    title="Number of Temperature Measurements per Heat (truncated at 20)",
    xlabel="Number of Measurements",
    ylabel="Number of Heats",
)
plt.tight_layout()
plt.show()

print(meas_per_heat.describe().round(1))
print(f"\nHeats with only 1 measurement: {(meas_per_heat == 1).sum():,}  "
      f"({(meas_per_heat == 1).mean():.1%})")
print(f"Heats with ≥ 3 measurements:     {(meas_per_heat >= 3).sum():,}  "
      f"({(meas_per_heat >= 3).mean():.1%})")

### Tapping temperature — target variable definition

The tapping temperature is defined as the last chronological measurement of each heat.
The distribution is narrow (σ = 13.9 °C), with 68% of heats finishing between 1636 °C
and 1664 °C — consistent with a well-controlled process targeting ~1650 °C.

This tight spread sets a demanding baseline: a `DummyRegressor` predicting the mean
already achieves MAE ≈ 11 °C. A model suitable for operational use should target MAE < 5 °C.

Outliers outside the 1550–1750 °C range are flagged for review later.

In [ ]:
tapping_temp = (
    eaf_temp
    .sort_values("DATETIME")
    .groupby("HEATID", as_index=False)
    .last()
    [["HEATID", "DATETIME", "TEMP", "VALO2_PPM"]]
    .rename(columns={
        "TEMP": "TEMP_TAPPING",
        "VALO2_PPM": "O2_PPM_TAPPING",
        "DATETIME": "DATETIME_TAPPING",
    })
)

fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(
    tapping_temp["TEMP_TAPPING"].dropna(),
    bins=60,
    edgecolor="white",
    linewidth=0.4,
)

mean_t = tapping_temp["TEMP_TAPPING"].mean()
std_t  = tapping_temp["TEMP_TAPPING"].std()

ax.axvline(mean_t, color="red",    linestyle="--", linewidth=1.2, label=f"Mean: {mean_t:.0f}°C")
ax.axvline(mean_t - std_t, color="orange", linestyle=":", linewidth=1.2, label=f"±1σ: {std_t:.1f}°C")
ax.axvline(mean_t + std_t, color="orange", linestyle=":", linewidth=1.2)

ax.set(
    title="Tapping temperature distribution (last measurement per heat)",
    xlabel="Temperature (°C)",
    ylabel="Number of heats",
)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean:            {mean_t:.1f} °C")
print(f"Std. deviation:  {std_t:.1f} °C  - model benchmark (DummyRegressor MAE)")
print(f"Minimum:         {tapping_temp['TEMP_TAPPING'].min():.0f} °C")
print(f"Maximum:         {tapping_temp['TEMP_TAPPING'].max():.0f} °C")


In [ ]:
print(f"Heats with tapping temperature: {len(tapping_temp):,}")
print(f"\n{tapping_temp[['TEMP_TAPPING', 'O2_PPM_TAPPING']].describe().round(1)}")

### Tapping temperature — temporal stability

Weekly mean and ±1σ band over the full 1305-day period. The global mean is shown
as a reference line.

The series splits into two distinct regimes:

- **2015–2017-04:** stable operation, mean hovering around 1650 °C with minor variation.

- **2017-05 to 2017-07:** complete data gap (~3 months), consistent with a possible planned refractory
  relining — a scheduled maintenance event with limited useful life of the EAF lining.
  
- **2017-08 onward:** mean shifts to ~1655–1660 °C and the weekly ±1σ band widens,
  suggesting a post event stabilization period followed by operation at a slightly
  higher target temperature.

Given that σ = 13.9 °C, a shift of +5–10 °C is small in absolute terms but non-negligible
relative to natural process variability.

**Decision:** train on the full dataset and monitor whether pre and post-event heats
concentrate systematically different prediction errors during evaluation.

In [ ]:
weekly_temp = (
    tapping_temp
    .set_index("DATETIME_TAPPING")["TEMP_TAPPING"]
    .resample("W")
    .agg(["mean", "std"])
)

fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(weekly_temp.index, weekly_temp["mean"], linewidth=0.9, label="Weekly mean")
ax.fill_between(
    weekly_temp.index,
    weekly_temp["mean"] - weekly_temp["std"],
    weekly_temp["mean"] + weekly_temp["std"],
    alpha=0.25,
    label="±1σ weekly",
)

ax.axhline(mean_t, color="red", linestyle="--", linewidth=0.8, label=f"Global mean ({mean_t:.0f}°C)")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=90, ha="right")

ax.set(
    title="Tapping Temperature — Weekly Mean (2015–2018)",
    xlabel="Date",
    ylabel="°C",
)
ax.legend()
plt.tight_layout()
plt.show()

### Tapping temperature — year-over-year comparison

Median and IQR are nearly identical across all four years, confirming the stability
observed in the weekly series. Extreme outliers (below 1500 °C and above 1750 °C)
appear in every year rather than clustering in a specific period, which is consistent
with isolated events — aborted heats or sensor faults — rather than a systemic issue.

**Conclusion:** a chronological train/test split is valid.
The process is stable enough that a model trained on 2015–2017 data can be evaluated
on 2018 without significant systematic bias. The post-event shift is real but small
enough not to invalidate this strategy.

In [ ]:
tapping_temp["year"] = tapping_temp["DATETIME_TAPPING"].dt.year

fig, ax = plt.subplots(figsize=(8, 4))
tapping_temp.boxplot(column="TEMP_TAPPING", by="year", ax=ax, grid=False)

ax.set(
    title="Tapping temperature by year",
    xlabel="Year",
    ylabel="Temperature (°C)",
)
fig.suptitle("")
plt.tight_layout()
plt.show()

print(
    tapping_temp.groupby("year")["TEMP_TAPPING"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(1)
)